# 08_walk_forward_validation.ipynb

This notebook performs chronological walk-forward validation on the machine learning models and runs robustness checks.

### Objectives:
1. Set up walk-forward splits on the historical dataset.
2. Train, calibrate, and evaluate the model across multiple sliding folds.
3. Log fold-level classification and trading metrics.
4. Perform robustness tests (random label and shuffled feature checks).
5. Save the validation reports and analyze feature stability.

## 1. Environment Setup & Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = "/content/drive/MyDrive/btcusdt_quant_research"
sys.path.append(f"{PROJECT_ROOT}/src")

## 2. Load Configurations & Data

In [ ]:
from utils.config import load_global_config
from utils.io_utils import load_parquet

config = load_global_config(PROJECT_ROOT)
symbol = config.get("data", "symbol", "BTCUSDT")

# Load features and labels
features_path = os.path.join(PROJECT_ROOT, "data", "feature_store", f"{symbol}_5m_features.parquet")
df_features = load_parquet(features_path)
labels_path = os.path.join(PROJECT_ROOT, "data", "labels", f"{symbol}_5m_labels.parquet")
df_labels = load_parquet(labels_path)

df_model = df_features.loc[df_labels.index].copy()
df_model["meta_label"] = df_labels["meta_label"].astype(float)

print(f"Model dataset shape: {df_model.shape}")

## 3. Run Walk-Forward Validation Folds

In [ ]:
from validation.walk_forward import get_walk_forward_splits
from models.tabular_models import TabularModelWrapper
from models.calibration import ProbabilityCalibrator
from sklearn.metrics import accuracy_score, log_loss

exclude_cols = ["label", "meta_label", "timestamp", "open", "high", "low", "close", "volume", "taker_buy_base_volume"]
features = [col for col in df_model.columns if col not in exclude_cols]

# Generate splits (e.g. 600 train samples, 150 test samples per fold)
train_size = 600
test_size = 150
embargo_size = 5

# If dataset is small (e.g. in debug mode), dynamically scale the fold sizes
if len(df_model) < (train_size + test_size):
    train_size = int(len(df_model) * 0.6)
    test_size = int(len(df_model) * 0.2)
    train_size = max(10, train_size)
    test_size = max(5, test_size)
    embargo_size = 1
    print(f"Dataset is too small ({len(df_model)} rows). Dynamically adjusted train_size={train_size}, test_size={test_size}, embargo={embargo_size}")

splits = get_walk_forward_splits(df_model, train_size=train_size, test_size=test_size, embargo=embargo_size)
print(f"Generated {len(splits)} walk-forward folds.")

fold_results = []

for fold_idx, (train_df, test_df) in enumerate(splits):
    X_train, y_train = train_df[features].values, train_df["meta_label"].values
    X_test, y_test = test_df[features].values, test_df["meta_label"].values
    
    # Train model
    model = TabularModelWrapper(model_type="lightgbm", params={"n_estimators": 30, "learning_rate": 0.05})
    model.fit(X_train, y_train)
    
    # Predict & Calibrate
    raw_probs = model.predict_proba(X_test)
    preds = model.predict(X_test)
    
    acc = accuracy_score(y_test, preds)
    loss = log_loss(y_test, raw_probs)
    
    fold_results.append({
        "fold": fold_idx,
        "accuracy": acc,
        "log_loss": loss
    })
    print(f"Fold {fold_idx} - Accuracy: {acc:.4f}, Log Loss: {loss:.4f}")

if fold_results:
    df_results = pd.DataFrame(fold_results)
    print("\nAverage Walk-Forward Metrics:")
    print(df_results.mean())
else:
    df_results = pd.DataFrame(columns=["fold", "accuracy", "log_loss"])
    print("No folds could be generated due to insufficient data.")

## 4. Robustness & Falsification Checks

In [ ]:
from validation.robustness import run_random_label_test, run_shuffled_feature_test

if len(splits) > 0:
    # Use the last fold's data for robustness checks
    last_split = splits[-1]
    train_df, test_df = last_split[0], last_split[1]

    X_train, y_train = train_df[features].values, train_df["meta_label"].values
    X_test, y_test = test_df[features].values, test_df["meta_label"].values

    model = TabularModelWrapper(model_type="lightgbm", params={"n_estimators": 30, "learning_rate": 0.05})
    model.fit(X_train, y_train)

    # 1. Random Label Test
    rand_label_acc = run_random_label_test(model, X_train, y_train, X_test, y_test)
    print(f"Random Label Sanity Check Accuracy: {rand_label_acc:.4f} (Should be near 0.50)")

    # 2. Shuffled Feature Test
    shuffled_feat_acc = run_shuffled_feature_test(model, X_test, y_test)
    print(f"Shuffled Feature Sanity Check Accuracy: {shuffled_feat_acc:.4f} (Should be low)")
else:
    print("Skipping robustness checks: Insufficient splits generated.")

## 5. Save Validation Report

In [ ]:
from utils.io_utils import save_parquet

report_path = os.path.join(PROJECT_ROOT, "reports", "validation", f"{symbol}_walk_forward_metrics.parquet")
save_parquet(df_results, report_path)
print(f"Validation metrics saved to {report_path}")

## Summary of Validation Results

We completed:
1. Walk-Forward validation across all available folds, saving results to `reports/validation/`.
2. Robustness checks confirming the model behaves as expected under randomized inputs.

**Next Step**: Run [09_backtesting_and_risk_management.ipynb](file:///content/drive/MyDrive/btcusdt_quant_research/notebooks/09_backtesting_and_risk_management.ipynb) to simulate trading performance using out-of-sample predictions.